# Lab 4

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Filtering

At the highest level, filtering an analog signal $f(t)$ with a filter $H(\omega)$ is computed by first computing a transform $f(t) \leftrightarrow F(\omega)$, then applying the filter $Y(\omega) = H(\omega) F(\omega)$, and finally getting the output $y(t) \leftrightarrow Y(\omega)$.

### Exercise 1

In lecture, we talked about ideal low pass, high pass and bandpass filters. However, it turns out that these filters are not actually physically realizable, for reasons that we'll explore in this lab.

#### Code

1. Design an ideal low pass filter `H` with $H(\omega) = 1$ for $\omega < 100 \frac{rads}{s}$. `H` should contain the theoretical rfft of your filter with a sampling rate of $400 \times 2\pi$ kHz and an array length of 400. Plot the frequency response of this filter.

2. Use the provided `freq_resp` function to evaluate and plot the approximate "analog" frequency response of your filter.

3. Use the provided `lpf_butter` function to generate a fourier-domain representation of a 4th order butterworth filter - call this `H_butter`. Plot the frequency response on the same plot as (2).

#### Written

1. How does the ideal low pass filter perform in comparison to the butterworth filter?

2. For analog signals, you saw that $Y(\omega) = H(\omega) X(\omega)$. Why do you think that this didn't work as well as you may have expected for digital signals? *Hint: what assumptions do you make when filtering in the analog domain?*

In [ ]:
##### DO NOT EDIT THIS CODE BOX #####
from scipy import signal

# omegas: array of omega values to evaluate freq response at
# H: your filter in array
# returns H(omega) for each omega value as a list
def freq_resp(H, label="$H(\omega)$"):
    # feel free to play around with the upsample rate - an analog filter would have an infinite sampling rate!
    upsample_rate = 100000;
    h = np.pad(np.fft.irfft(H), (0, upsample_rate - len(H)))
    H = np.fft.rfft(h)
    plt.xlim(0,10000)
    plt.plot(np.abs(H), label=label)

def lpf_butter(cutoff, sr, length, order=4):
    b, a = signal.butter(order, cutoff/sr, 'low')
    impulse = np.zeros(length)
    impulse[0] = 1
    return np.fft.rfft(signal.lfilter(b, a, impulse))

In [11]:
# WRITE YOUR CODE HERE

Write your written responses in this box.

## Convolution

To avoid many of the issues you discovered in the previous exercise, we instead use **convolution** to apply arbitrary filters to time domain signals. We've given you a few exercises below to practice convolving signals.

### Exercise 2

#### Written

1. $h = [0, 0.5, 1, 0.5, 0], x = [1, 2, 3, 4, 5], y = h * x = ?$

2. $h = [0, 0, 1, 1, 0, 0, 0, 0], x = [0, 1, 0, 1, 0, 1], y = h * x = ?$

3. $h = [1, 1, 0, 0, 0], x = [1, 0, 0, 0, 0, 0, 0, 0], y = h * x = ?$

Write your written responses in this box.

### Exercise 3

You can use `np.convolve` to convolve signals in numpy. In this exercise, you'll solve a very common issue in audio signal processing.

#### Code

1. Run the given code box below, and listen to the audio. You should hear a sine tone with random noise in the background.

2. Determine where the audio frequencies of interest are located, using a fourier transform. What filter(s) would you use to remove the noise from your signal?

3. Use our helper function `butter` to construct time domain representations of the Butterworth filter(s) you need to remove noise from your signal. Then, use `np.convolve` to apply your filters to the `noisy_audio`, and call your filtered signal `denoised_audio`.

4. Compare the frequency domain representations of `noisy_audio`, `denoised_audio` and `clean_audio` with a plot.

#### Written

1. Which filters did you choose, and why?

2. Why does `noisy_audio` not match `clean_audio` perfectly? Your answer should include some discussion of the characteristics of the Butterworth filter (*hint: see Exercise 1*).

3. Which parameters would you change to improve your filter performance, if you were required to use a Butterworth filter?

4. For your chosen filter, what types of noise do you think it would be best at removing? What types of noise would it be the worst at removing?

In [ ]:
##### DO NOT EDIT THIS CODE BOX #####
from IPython.display import Audio
from scipy import signal

def butter(cutoff, sr, length, type='low', order=4):
    b, a = signal.butter(order, cutoff/sr, type)
    impulse = np.zeros(length)
    impulse[0] = 1
    return np.fft.rfft(signal.lfilter(b, a, impulse))

def mean_sq(s1, s2):
    return np.sum(np.abs(s1 - s2))

duration = 5
sr = 22000

t = np.linspace(0, duration, sr * duration)
clean_audio = np.sin(440 * 2 * np.pi * t)
noise = np.random.rand(t.shape[0]) - 0.5
noisy_audio = clean_audio + noise
plt.plot(abs(np.fft.rfft(noisy_audio)))
plt.xlim(0,5000)
Audio(noisy_audio, rate=sr)
# frequency of the peak = 2200 * sr/(sr*duration) Hz = 440 Hz

In [12]:
# Write your code in this box

Write your written response in this box.

### Exercise 4

Most computer vision systems nowadays are built on top of Convolutional Neural Networks (CNNs). Just like you can apply 1D filters to audio signals, you can apply 2D filters to images. CNNs use "machine learning" (we're treating this as a bit of a black box since ML isn't in the scope of this class) to design filters (just like you designed filters in the exercises above), and apply these filters using convolution.

For this exercise, we'll give you the following "spatial domain" (i.e not frequency domain) representation for "lowpass" and "highpass" filters:

$LPF = \begin{bmatrix}
\frac{1}{N^2} & \frac{1}{N^2} & ... & \frac{1}{N^2}\\
... & ... & ... & ...\\
\frac{1}{N^2} & \frac{1}{N^2} & ... & \frac{1}{N^2}\\
\end{bmatrix}$ where $LPF$ is an $N \times N$ matrix.

$HPF = \begin{bmatrix}
0 & -\frac{1}{4} & 0\\
-\frac{1}{4} & 2 & -\frac{1}{4}\\
0 & -\frac{1}{4} & 0
\end{bmatrix}$

#### Written

1. Apply a $10 \times 10$ low pass filter to your image `img_gs` using `scipy.signal.conv2d`. What do you notice about the behavior? How would you qualititatively describe a low pass filter for a 2D signal?

2. Apply a high pass filter to `img_gs`. What do you notice about the behavior? How would you qualitalitatively describe a high pass filter for a 2D signal?

3. What happens if you apply a $3 \times 3$ low pass filter to your image, then apply your high pass filter to your image? Is this an expected behavior? If so, why?

In [ ]:
import skimage

img_url = "http://publish.illinois.edu/inspire-illinois/files/2014/04/UIUC-logo.gif"
img = skimage.io.imread(img_url)[0]

# convert to greyscale, and reduce image size for speed
img_gs = np.mean(img, axis=2)[200:400, 200:400]
plt.imshow(img_gs, cmap="gray");

In [ ]:
#### WRITE YOUR CODE HERE ####

Write your written response here.

### Exercise 5
Please fill out the survey below to give feedback on the course so far. https://forms.gle/S1ZNQwyx5XfzjpXZ9

In [ ]:
secret_password = ...